Hybrid Search Langchain

In [1]:
from langchain_community.retrievers import PineconeHybridSearchRetriever
from pinecone import Pinecone,ServerlessSpec
import os
from dotenv import load_dotenv
load_dotenv()

C:\Users\Rishikesh Reddy\AppData\Local\Temp\ipykernel_30188\569930749.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import PineconeHybridSearchRetriever
d:\AI\LangChain\venv\lib\site-packages\pinecone\data\index.py:1: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


True

In [2]:
api_key=os.getenv("PINECONE_API_KEY")
index_name="hybrid-search-langchain"

#initialize  the pinecone client
pc=Pinecone(api_key=api_key)

if index_name not in pc.list_indexes().names():
    pc.create_index(name=index_name,
                    dimension=384, # As we are using the sentence transformer(for dense vector)
                    metric='dotproduct',#(for sparse vector)
                    spec=ServerlessSpec(cloud='aws',region='us-east-1'),
                    )

In [3]:
index=pc.Index(index_name)
index

In [4]:
# For Dense vectors
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8380.31it/s]


HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [5]:
# For Sparse Vector
from pinecone_text.sparse import BM25Encoder
bm25_encoder=BM25Encoder().default()
bm25_encoder

In [6]:
sentences=[
    "In 2023, I visited Paris",
    "In 2022, I visited New York",
    "In 2021, I visited New Orleans"
]

# tfidf values on thses sentence
bm25_encoder.fit(sentences)

#store the values to a json file
bm25_encoder.dump('bm25_values.json')

# load to Bm25Encoder Object
bm25_encoder=BM25Encoder().load("bm25_values.json")

100%|██████████| 3/3 [00:00<00:00, 103.98it/s]


In [7]:
retriever=PineconeHybridSearchRetriever(embeddings=embeddings,sparse_encoder=bm25_encoder,index=index)
retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x00000202036AF1F0>, index=<pinecone.data.index.Index object at 0x00000202036AF7C0>)

In [8]:
# INserting into index of the Pinecone
retriever.add_texts(
    [
    "In 2023, I visited Paris",
    "In 2022, I visited New York",
    "In 2021, I visited New Orleans"
]
)

100%|██████████| 1/1 [00:02<00:00,  2.19s/it]


In [10]:
retriever.invoke("What city did i visit first")

[Document(metadata={'score': 0.23936905}, page_content='In 2021, I visited New Orleans'),
 Document(metadata={'score': 0.232817784}, page_content='In 2022, I visited New York'),
 Document(metadata={'score': 0.212499827}, page_content='In 2023, I visited Paris')]